# NVFlare CLI Tutorial

This notebook walks through a working end-to-end NVFlare CLI workflow against a running two-client POC system. It replaces the older Job CLI tutorial, which focused on deprecated job-template commands.

The notebook itself does not start or stop POC services. Start POC from a terminal first, run the notebook commands, then stop POC from the terminal when finished.


## Terminal Prerequisite

Run these commands in a terminal before running the notebook cells:

```bash
nvflare poc config --pw /tmp/nvflare_cli_tutorial
nvflare poc prepare -n 2 --force
nvflare poc start --timeout 60
```

`poc prepare` creates the local startup kits, registers the Project Admin startup kit, and makes it active for `nvflare job` and `nvflare system` commands. After finishing the tutorial, stop the POC system from the terminal:

```bash
nvflare poc stop
```


## Notebook Values

The job export and result download use `/tmp` so rerunning the tutorial does not write into the repository. A fresh submit token is generated for each notebook run.


In [ ]:
from uuid import uuid4

JOB_FOLDER = "/tmp/nvflare/jobs/job_config/hello-pt"
STUDY_NAME = f"cli-tutorial-{uuid4().hex[:8]}"
RESULT_DIR = f"/tmp/nvflare_cli_tutorial/results-{uuid4().hex[:8]}"
SUBMIT_TOKEN = f"nvflare-cli-tutorial-hello-pt-{uuid4().hex}"
ABORT_SUBMIT_TOKEN = f"nvflare-cli-tutorial-abort-{uuid4().hex}"

print(f"JOB_FOLDER={JOB_FOLDER}")
print(f"STUDY_NAME={STUDY_NAME}")
print(f"RESULT_DIR={RESULT_DIR}")
print(f"SUBMIT_TOKEN={SUBMIT_TOKEN}")
print(f"ABORT_SUBMIT_TOKEN={ABORT_SUBMIT_TOKEN}")


## Check the CLI Surface

Use `--help` and `--schema` to verify the active command interface. In a source checkout, the version flag can reflect local Git tag state, so this tutorial checks the commands directly.


In [ ]:
!nvflare --help
!nvflare job --help
!nvflare job submit --schema


## Discover Recipes

The current workflow is recipe-based: export or create a job folder, then submit it with the CLI. `recipe list` and `recipe show` are the replacement for the old job-template discovery flow.


In [ ]:
!nvflare recipe list
!nvflare recipe show fedavg


## Verify Startup-Kit Configuration

These commands should show the Project Admin startup kit registered by `nvflare poc prepare`. If there is no active startup kit, return to the terminal prerequisite and run `poc prepare` again.


In [ ]:
!nvflare config list
!nvflare config inspect


## Verify the Running FL System

Now confirm the server and both POC clients are reachable. This is the first meaningful check that the notebook is connected to a live FL system.


In [ ]:
!nvflare system status
!nvflare system resources server
!nvflare system resources client site-1 site-2
!nvflare system version --site all


## Configure System Logging

`system log-config` changes the runtime log level or log mode for the server and client sites. This tutorial applies `INFO`, which is the normal operating level and is safe to run repeatedly.


In [ ]:
!nvflare system log-config INFO --site all


## Work with Study Scope

A study scopes site membership, user access, and job history. This notebook creates a temporary named study with the two POC client sites, submits the job to that study, and removes the study after downloading the result. Production users follow the same pattern with collaboration-specific study names and site membership.


In [ ]:
!nvflare study --help
!nvflare study list
!nvflare study register {STUDY_NAME} --site-org nvidia:site-1,site-2
!nvflare study show {STUDY_NAME}


## Export a Real Job Folder

This tutorial uses `examples/hello-world/hello-pt`, a PyTorch FedAvg job. The recipe-level export command enables job log streaming so the later `job logs` cell can retrieve server and client logs. It also uses synthetic data so the tutorial can run without downloading CIFAR-10. The `cd` command keeps the export self-contained: `job.py` runs from its example directory so it can find `client.py` and `model.py`.


In [ ]:
!cd ../hello-world/hello-pt && python job.py --export --export-dir /tmp/nvflare/jobs/job_config --enable_log_streaming --synthetic_data --train_size 2048 --test_size 256 --num_rounds 2 --epochs 1 --batch_size 64 --num_workers 0
!find {JOB_FOLDER} -maxdepth 3 -type f | sort | head -30


## Submit the Job

`job submit` returns immediately with a `job_id`. The submit token makes this operation safe to retry for the same job content and submitter.


In [ ]:
submit_result = !nvflare --format json job submit -j {JOB_FOLDER} --study {STUDY_NAME} --submit-token {SUBMIT_TOKEN}
print("\n".join(submit_result))


In [ ]:
import json

submit_response = json.loads("\n".join(submit_result))
if submit_response.get("status") != "ok":
    raise RuntimeError(json.dumps(submit_response, indent=2))

submit_data = submit_response.get("data", {})
JOB_ID = submit_data.get("job_id") or submit_data.get("existing_job_id")
if not JOB_ID:
    raise RuntimeError(json.dumps(submit_response, indent=2))
print(f"JOB_ID={JOB_ID}")


## Configure Job Logging

`job log` is an alias for `job log-config`. It changes runtime logging for an active job in the selected study. Because this tutorial job is intentionally short, the command may find that the job has already completed; that is handled as a non-fatal timing case.


In [ ]:
!nvflare job log {JOB_ID} INFO --study {STUDY_NAME} --site all


## Recover the Job ID from the Submit Token

If automation loses the original submit response, it can recover the submitted job by querying with the same submit token.


In [ ]:
lookup_result = !nvflare --format json job list --study {STUDY_NAME} --submit-token {SUBMIT_TOKEN}
print("\n".join(lookup_result))


## List, Monitor, and Wait

Use `job list` for recent job history, `job stats` while the job is running, `job monitor` for interactive progress, and `job wait` for automation that needs a final command result. Each command below passes `--study {STUDY_NAME}` so the job lookup happens in the intended study. The tutorial job is intentionally small, so it can finish before the stats command samples it on fast systems or when cells are run slowly.


In [ ]:
!nvflare job list --study {STUDY_NAME} -m 5


In [ ]:
stats_result = !nvflare --format json job stats {JOB_ID} --study {STUDY_NAME} --site all
print("\n".join(stats_result))

stats_response = json.loads("\n".join(stats_result))
if stats_response.get("status") != "ok":
    # Older servers reported a completed job as an INTERNAL_ERROR with an unstructured message.
    not_running = stats_response.get("error_code") == "JOB_NOT_RUNNING" or (
        stats_response.get("error_code") == "INTERNAL_ERROR" and "is not running" in stats_response.get("message", "")
    )
    if not_running:
        print("The job completed before live stats were sampled; continue with monitor/wait for terminal status.")
    else:
        raise RuntimeError(json.dumps(stats_response, indent=2))


In [ ]:
!nvflare job monitor {JOB_ID} --study {STUDY_NAME} --timeout 120 --interval 5


In [ ]:
wait_result = !nvflare --format json job wait {JOB_ID} --study {STUDY_NAME} --timeout 600 --interval 5
print("\n".join(wait_result))

wait_response = json.loads("\n".join(wait_result))
if wait_response.get("status") != "ok":
    raise RuntimeError(json.dumps(wait_response, indent=2))


## Inspect the Completed Job

After the job reaches a terminal state, inspect its metadata and retrieve streamed server/client logs.


In [ ]:
!nvflare job meta {JOB_ID} --study {STUDY_NAME}


In [ ]:
!nvflare job logs {JOB_ID} --study {STUDY_NAME} --sites all --tail 80


## Download Job Results

Download the server-side job artifacts and inspect the downloaded files.


In [ ]:
!nvflare job download {JOB_ID} --study {STUDY_NAME} --output-dir {RESULT_DIR} --force
!find {RESULT_DIR} -maxdepth 3 -type f | sort | head -50


## Abort a Running Job

Submit a second job in the same study and abort it. The abort demo uses a longer synthetic run so the job is still active when `job abort` is issued. `job wait` reports `JOB_ABORTED` for an aborted terminal state; that is the expected result here. The only Python below extracts the generated job id so the following CLI commands can reference it.


In [ ]:
!cd ../hello-world/hello-pt && python job.py --export --export-dir /tmp/nvflare/jobs/job_config --enable_log_streaming --synthetic_data --train_size 8192 --test_size 512 --num_rounds 20 --epochs 1 --batch_size 64 --num_workers 0

abort_submit_result = !nvflare --format json job submit -j {JOB_FOLDER} --study {STUDY_NAME} --submit-token {ABORT_SUBMIT_TOKEN}
print("\n".join(abort_submit_result))

abort_response = json.loads("\n".join(abort_submit_result))
if abort_response.get("status") != "ok":
    raise RuntimeError(json.dumps(abort_response, indent=2))

abort_data = abort_response.get("data", {})
ABORT_JOB_ID = abort_data.get("job_id") or abort_data.get("existing_job_id")
if not ABORT_JOB_ID:
    raise RuntimeError(json.dumps(abort_response, indent=2))
print(f"ABORT_JOB_ID={ABORT_JOB_ID}")

!nvflare job abort {ABORT_JOB_ID} --study {STUDY_NAME} --force
!nvflare job wait {ABORT_JOB_ID} --study {STUDY_NAME} --timeout 120 --interval 5
!nvflare job meta {ABORT_JOB_ID} --study {STUDY_NAME}


## Clean Up the Study

After downloading the result and aborting the second job, remove the remote job records and delete the temporary study created for this notebook. The downloaded files remain in `RESULT_DIR`.


In [ ]:
!nvflare job delete {ABORT_JOB_ID} --study {STUDY_NAME} --force
!nvflare job delete {JOB_ID} --study {STUDY_NAME} --force
!nvflare study remove {STUDY_NAME}


## Other CLI Command Groups

The end-to-end workflow above used `config`, `study`, `system`, `recipe`, and `job`. The same `nvflare` CLI also includes command groups for distributed provisioning, package assembly, and deployment preparation. The commands below are read-only help commands so they are safe to run in this notebook.


In [ ]:
!nvflare cert --help
!nvflare package --help
!nvflare deploy prepare --help


## Quick Migration Reference

| Older tutorial command | Current workflow |
| --- | --- |
| `nvflare job list-templates` | `nvflare recipe list` and `nvflare recipe show <recipe>` |
| `nvflare job create` | Create/export a job folder with Job Recipe API code or an example script, then submit it |
| `nvflare job show-variables` | Inspect the recipe, example code, or generated job folder directly |
| Repeating startup-kit paths in each command | POC: `nvflare poc prepare` registers the Project Admin kit. Production: register a real admin startup kit with `nvflare config add/use`. |
| Implicit default job scope | Pass `--study <study_name>` on `job submit/list/monitor/wait/meta/logs/stats/download/clone/abort/delete` when working outside the default study. |
| Admin Console-only job operations | `nvflare job submit/list/monitor/wait/meta/logs/stats/download/clone/abort/delete` |
| Admin Console-only system checks | `nvflare system status/resources/version/log-config/restart/shutdown` |
